# ResNet18 Transfer Learning and Controlled Fine-Tuning

This section presents the Member 4 contribution: a ResNet18 transfer-learning model for 30-class tree-species classification. It covers model adaptation, staged fine-tuning, controlled learning-rate comparison, validation-based model selection, final test evaluation, and per-class analysis.

> **Submission note:** This development notebook section must be merged into the group's single final Project Notebook. It must not be submitted as a separate notebook.

## 1. Research question and contribution

The experiment addresses the following question:

**Can partial fine-tuning of an ImageNet-pretrained ResNet18 substantially improve tree-species classification compared with training only the final classifier?**

TorchVision provides the ResNet18 architecture and ImageNet weights. The project-specific contribution includes:

- replacing the original 1,000-class classifier with a 30-class layer;
- implementing `head`, `layer4`, and `all` trainable scopes;
- applying ImageNet normalisation without changing the custom CNN default;
- preserving frozen BatchNorm running statistics;
- loading the best frozen-head checkpoint before Layer4 fine-tuning;
- conducting a controlled learning-rate comparison;
- calculating overall, Top-5, and per-class metrics; and
- generating reproducible result tables and figures.

In [ ]:
import csv
import json
from pathlib import Path

from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'report').exists():
    ROOT = ROOT.parent

assert (ROOT / 'report/tables/resnet18_experiments.csv').is_file()
assert (ROOT / 'report/tables/resnet18_final_test_metrics.csv').is_file()
print(f'Project root: {ROOT}')

## 2. Data and preprocessing

All experiments use the fixed, group-aware Part 2 splits. No data split was regenerated during the ResNet18 experiments.

| Split | Number of images |
|---|---:|
| Training | 5,070 |
| Validation | 1,094 |
| Test | 1,075 |
| **Total** | **7,239** |

Images are converted to RGB and resized to 224 x 224. Training data uses the shared augmentation pipeline, while validation and test preprocessing is deterministic. Because ResNet18 uses ImageNet-pretrained weights, inputs use the matching channel statistics:

```text
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]
```

The class mapping and all three split CSV files remain identical across the controlled experiments.

## 3. Model and fine-tuning strategy

ResNet18 uses residual blocks that learn a residual mapping and add it to the block input: $y = F(x) + x$. The skip connection improves gradient flow and makes deeper convolutional networks easier to optimise.

The model is initialised with `ResNet18_Weights.DEFAULT`, and the final fully connected layer is replaced by a 30-class layer. The adapted network has **11,191,902 total parameters**.

Two required stages are evaluated:

1. **Frozen head stage:** all backbone parameters are frozen and only the final classifier is trained.
2. **Layer4 fine-tuning stage:** the final residual block and classifier are trainable. The stage starts from the best frozen-head checkpoint.

Frozen BatchNorm modules remain in evaluation mode during training, preventing unintended updates to their running statistics.

In [ ]:
from src.models.resnet18 import build_resnet18, count_parameters

parameter_counts = {}
for scope in ('head', 'layer4', 'all'):
    model = build_resnet18(
        num_classes=30,
        trainable_scope=scope,
        pretrained=False,
    )
    parameter_counts[scope] = count_parameters(model)

parameter_counts

Expected parameter counts:

| Scope | Total parameters | Trainable parameters |
|---|---:|---:|
| Head | 11,191,902 | 15,390 |
| Layer4 | 11,191,902 | 8,409,118 |
| All | 11,191,902 | 11,191,902 |

## 4. Controlled experimental setup

All runs use seed 42, batch size 16, weight decay $10^{-4}$, 224 x 224 inputs, the same augmentation, the same ImageNet normalisation, and the same train/validation/test splits. The comparison changes only the trainable scope or the Layer4 learning rate.

The frozen-head stage trains for 5 epochs. Both Layer4 experiments train for 10 epochs from the same frozen checkpoint. Checkpoints are selected using validation performance; the test set is not used during training or hyperparameter selection.

In [ ]:
experiment_path = ROOT / 'report/tables/resnet18_experiments.csv'
with experiment_path.open(newline='', encoding='utf-8') as file:
    experiments = list(csv.DictReader(file))

header = '| Experiment | Scope | LR | Best epoch | Val accuracy | Val Macro-F1 |'
separator = '|---|---|---:|---:|---:|---:|'
rows = [header, separator]
for row in experiments:
    rows.append(
        f"| {row['experiment']} | {row['trainable_scope']} | "
        f"{float(row['learning_rate']):.0e} | {row['best_epoch']} | "
        f"{float(row['val_accuracy']):.2%} | {float(row['val_macro_f1']):.2%} |"
    )
display(Markdown('\n'.join(rows)))

## 5. Training behaviour

![ResNet18 training and validation curves](../report/figures/resnet18/resnet18_training_curves.png)

The frozen-head model improves steadily but reaches a lower validation ceiling. Unfreezing Layer4 produces a large performance gain. The $3\times10^{-5}$ run improves more gradually than the $10^{-4}$ run and achieves the highest validation accuracy.

For the selected run, training accuracy reaches 98.22% and validation accuracy reaches 96.44%. The remaining gap indicates mild overfitting, but the validation curve remains stable. The lowest validation loss occurs at epoch 7, while the highest validation accuracy occurs at epoch 10. The predeclared checkpoint rule uses validation accuracy.

## 6. Validation comparison and model selection

![ResNet18 validation comparison](../report/figures/resnet18/resnet18_validation_comparison.png)

| Experiment | Validation Accuracy | Validation Macro-F1 | Validation Weighted-F1 |
|---|---:|---:|---:|
| Frozen head | 89.49% | 89.26% | 89.45% |
| Layer4, LR=1e-4 | 95.98% | 95.90% | 95.75% |
| **Layer4, LR=3e-5** | **96.44%** | **96.08%** | **96.25%** |

Partial fine-tuning improves validation accuracy by 6.95 percentage points over the frozen-head model. The $3\times10^{-5}$ run also provides the best Macro-F1 and Weighted-F1, so it is locked as the final model before test evaluation.

## 7. Final test results

The selected checkpoint is evaluated once on the 1,075-image test set. No hyperparameter or checkpoint is changed after observing these results.

| Metric | Test score |
|---|---:|
| Accuracy | **95.53%** |
| Macro Precision | 96.47% |
| Macro Recall | 95.29% |
| Macro-F1 | **95.39%** |
| Weighted Precision | 96.22% |
| Weighted Recall | 95.53% |
| Weighted-F1 | **95.39%** |
| Top-5 Accuracy | **100.00%** |

The validation-to-test accuracy gap is 0.91 percentage points, and the Macro-F1 gap is 0.69 percentage points. The small gaps indicate good held-out generalisation.

In [ ]:
test_metrics_path = ROOT / 'report/tables/resnet18_final_test_metrics.csv'
with test_metrics_path.open(newline='', encoding='utf-8') as file:
    final_test_metrics = next(csv.DictReader(file))

for metric in ('accuracy', 'macro_f1', 'weighted_f1', 'top5_accuracy'):
    print(f'{metric}: {float(final_test_metrics[metric]):.4f}')

## 8. Per-class analysis

![Final ResNet18 test F1 by species](../report/figures/resnet18/resnet18_test_per_class_f1.png)

Most species achieve high F1 scores, but the weakest class is `quercus_muehlenbergii`, with 100% precision, 47.06% recall, and 64.00% F1. This pattern means that predictions assigned to the class are reliable, but many true examples are assigned to other species.

`prunus_sargentii` shows the opposite pattern: 67.74% precision and 100% recall. The model retrieves all true examples but also incorrectly assigns examples from other classes to this category. These two classes should be prioritised in the group confusion-matrix and error-analysis section.

In [ ]:
per_class_path = ROOT / 'report/tables/resnet18_test_per_class_metrics.csv'
with per_class_path.open(newline='', encoding='utf-8') as file:
    per_class = list(csv.DictReader(file))

worst_five = sorted(per_class, key=lambda row: float(row['f1']))[:5]
header = '| Species | Precision | Recall | F1 | Support |'
separator = '|---|---:|---:|---:|---:|'
rows = [header, separator]
for row in worst_five:
    rows.append(
        f"| {row['class_name']} | {float(row['precision']):.2%} | "
        f"{float(row['recall']):.2%} | {float(row['f1']):.2%} | "
        f"{row['support']} |"
    )
display(Markdown('\n'.join(rows)))

## 9. Discussion

### Strengths

- Layer4 fine-tuning produces a large and consistent improvement over the frozen-head model.
- Accuracy, Macro-F1, and Weighted-F1 are close, indicating that strong performance is not limited to the largest classes.
- The validation-to-test gap is small, suggesting good generalisation.
- Top-5 accuracy reaches 100%, so the correct class is always among the five highest-scoring predictions in the final test set.

### Weaknesses and limitations

- A small number of visually similar species remain difficult, especially `quercus_muehlenbergii`.
- Training accuracy is higher than validation accuracy, indicating mild overfitting.
- The current Part 4 analysis does not separately quantify laboratory and field performance.
- The search covers a small controlled set of settings rather than an exhaustive hyperparameter sweep.
- Training time was not measured consistently across every run, so runtime comparisons would not be reliable.

### Future work

The final group analysis should add source-specific metrics, confusion matrices, representative error cases, and Grad-CAM visualisations. Targeted augmentation or class-aware sampling may help the weakest species, but such changes must be evaluated without reusing the final test set for model selection.

## 10. Reproducibility

The commands used for the three controlled experiments are:

```bash
python part4_resnet18.py --config configs/resnet18_frozen.yaml
python part4_resnet18.py --config configs/resnet18_layer4.yaml
python part4_resnet18.py --config configs/resnet18_layer4_lr3e5.yaml
```

Validation evaluation is the default:

```bash
python part4_evaluate_resnet18.py
```

The test command is recorded for reproducibility but must not be rerun for further tuning:

```bash
python part4_evaluate_resnet18.py --split test
```

Figures and tracked tables can be regenerated from saved result files without running a model:

```bash
python part4_plot_results.py
```

## 11. Conclusion

Partial fine-tuning is essential for this task. Training only the classifier reaches 89.49% validation accuracy, while Layer4 fine-tuning with a learning rate of $3\times10^{-5}$ reaches 96.44%. The locked model achieves 95.53% test accuracy and 95.39% test Macro-F1. The remaining errors are concentrated in a small number of species, providing clear targets for the group's error analysis and explainability work.